# Phase II: Data Curation, Exploratory Analysis and Plotting (5\%)

### Team Members:
- Mikhail Pevunov
- Joshua Moshes
- Brandon Diaz
- Alex Sun


## Part 1:
(1%) Expresses the central motivation of the project and explains the (at least) two key questions to be explored. Gives a summary of the data processing pipeline so a technical expert can easily follow along.

## Problem Motivation

Movie success is often influenced by how well it resonates with both critics and audiences, yet these two perspectives don’t always align. Understanding what drives the differences or similarities between audience and critic reception can provide insight into broader patterns in film evaluation. In this project, we aim to investigate whether certain genres tend to receive more favorable responses from critics or audiences, and whether there is a measurable correlation between genre, box office revenue, rating, audience score, and critics score. The question we are trying to answer is whether or not there is a correlation between genre, box office revenue, or rating and audience or critic score. By analyzing movies released from 2000-2025 using data from Rotten tomatoes, our goal is to uncover trends that reveal how critical acclaim and popular opinion intersect or diverge across different film genres. The key questions are as follows:

1. Is there a correlation between the genre to audience score and the genre to critics score?
2. Is there a correlation between the box office revenue to audience score and the box office revenue to critics score?
3. Is there a correlation between the rating to audience score and the rating to critics score?

Motivating sources:
- “Top Rated Movies — the Movie Database (TMDB).” The Movie Database, www.themoviedb.org/movie/top-rated?language=en-US. Accessed 26 Oct. 2025.
- “Most Popular Movies.” IMDb, www.imdb.com/chart/moviemeter/?ref_=chtmvm_nv_menu. Accessed 26 Oct. 2025.
- “Best Certified Fresh Movies Out Now in Theaters (2025).” Rotten Tomatoes, www.rottentomatoes.com/browse/movies_in_theaters/critics:certified_fresh~sort:popular. Accessed 26 Oct. 2025.



## Summary of the Data Processing Pipeline

1. Scraped raw data from Rotten Tomatoes - used selenium to get the full list of movies (because loading in more than the first 5 page was not possible on rotten tomatoes with just the urls) and beautifulsoup to get all the data from movies
2. Cleaned the data to prepare the data frame for visualization and analysis
3. Visualized using matplotlib and numpy

## Part 2:
(2\%) Obtains, cleans, and merges all data sources involved in the project.

In [2]:
!pip install google-colab-selenium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.0/512.0 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 51.6 MB/s eta 0:00:00


In [3]:
# We submitted the data.json file separately because it takes a while to run this code
import time
import json
import tempfile
import shutil

import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By

general_list = []
temp_dict = {}

def getFullRottenTomatoesList(numPages):
    """
    Gets list of rotten tomatoes pages and initializes browser

    Args:
    numPages (int): Number of pages to parse

    Returns: html of page
    """
    tmp_profile = tempfile.mkdtemp(prefix="chrome-profile-")
    opts = Options()
    opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument(f"--user-data-dir={tmp_profile}")
    opts.add_experimental_option("excludeSwitches", ["enable-logging"])

    browser = webdriver.Chrome(options=opts)
    browser._temp_profile_dir = tmp_profile

    browser.get('https://www.rottentomatoes.com/browse/movies_at_home')
    WebDriverWait(browser, 1000).until(EC.presence_of_element_located((By.ID, "onetrust-accept-btn-handler")))
    browser.find_element(By.ID, "onetrust-accept-btn-handler").click()

    i = 0
    while i < numPages:
        WebDriverWait(browser, 1000).until(EC.presence_of_element_located((By.CLASS_NAME, "discovery__actions")))
        time.sleep(3)
        browser.find_element(By.CLASS_NAME, 'discovery__actions').find_element(By.TAG_NAME, 'button').click()
        i += 1
    return browser.page_source

def parseRottenTomatoesPopularMoviesWithHTML(content):
    """
    Parses HTML and sends list of needed divs to parseMoveSquare

    Args:
    content (string): HTML of the content
    """
    soup = BeautifulSoup(content, "html.parser")

    divs = soup.find('div', class_="discovery-tiles__wrap").find_all("div", class_="flex-container")
    for div in divs:
        parseMoveSquare(div)

def parseMoveSquare(div):
    """
    Gets the content of each movie's square

    Args:
    div (string): div of the movie
    """
    global temp_dict
    url_of_movie = "https://www.rottentomatoes.com" + div.find('a').get('href')
    parseMovieUrl(url_of_movie)

def parseMovieUrl(url):
    """
    Given url of an individual movie, it saves it to the general list

    Args:
    url (string): url to parse
    """
    global temp_dict, general_list
    response = requests.get(url)
    html_content = response.content

    soup = BeautifulSoup(html_content, "html.parser")
    temp_dict["url"] = url

    potential_scores = soup.find('div', id="main-wrap").find_all('rt-text')
    dict_scores = {}
    for rt in potential_scores:
        if rt.get("slot") == "criticsScore":
            dict_scores['criticsScore'] = rt.text.replace("\n", '').replace(" ", '')
        if rt.get("slot") == "audienceScore":
            dict_scores['audienceScore'] = rt.text.replace("\n", '').replace(" ", '')
    temp_dict["scores"] = dict_scores

    rts = soup.find('div', class_="media-hero-wrap").find_all("rt-text")
    genres = []
    for rt in rts:
        if rt.get("slot") == "metadataGenre":
            genres.append(rt.text)
        if rt.get("slot") == "title":
            temp_dict['title'] = rt.text.replace("\n", '')
    temp_dict["genres"] = genres

    movie_info = soup.find('section', class_="media-info").find_all('div', class_="category-wrap")
    for movie_stat in movie_info:
        if "Box Office" in movie_stat.text:
            temp_dict["boxOffice"] = movie_stat.find_all('rt-text')[1].text
        if "Rating" in movie_stat.text:
            temp_dict["rating"] = movie_stat.find_all('rt-text')[1].text

    general_list.append(temp_dict)
    temp_dict = {}
    print(general_list)


def save_to_json(data, filename):
    """
    Shortcut to save quickly to json

    Args:
    data (list): Data to add to json
    filename (string): filename to add data to
    """
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

try:
    parseRottenTomatoesPopularMoviesWithHTML(getFullRottenTomatoesList(15))
except Exception as e:
    print(f"Exception: {e}")
    print(f"Number of movies: {len(general_list)}")
    save_to_json(general_list, "data.json")
    exit()

print(f"Number of movies: {len(general_list)}")
save_to_json(general_list, "data.json")

Exception: Message: session not created: Chrome instance exited. Examine ChromeDriver verbose log to determine the cause.; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#sessionnotcreatedexception
Stacktrace:
#0 0x5affd435e4ca <unknown>
#1 0x5affd3dace4b <unknown>
#2 0x5affd3de7919 <unknown>
#3 0x5affd3de3375 <unknown>
#4 0x5affd3e33fe6 <unknown>
#5 0x5affd3e33706 <unknown>
#6 0x5affd3df1c2a <unknown>
#7 0x5affd3df2931 <unknown>
#8 0x5affd4324cf9 <unknown>
#9 0x5affd4327cdc <unknown>
#10 0x5affd430df79 <unknown>
#11 0x5affd43288b5 <unknown>
#12 0x5affd42f59c3 <unknown>
#13 0x5affd434b228 <unknown>
#14 0x5affd434b403 <unknown>
#15 0x5affd435d463 <unknown>
#16 0x79e1dec04ac3 <unknown>

Number of movies: 0
Number of movies: 0


In [3]:
# Data cleaning:

import pandas as pd
import json

def clean_rotten_tomatoes_data(data_list):
    """
    Cleans scraped Rotten Tomatoes movie data and structures it into a pandas DataFrame.
    Note: removes rows with any NaN values.

    Args:
    data_list (list): A list of dictionaries, where each dictionary represents a movie
                      and contains keys like 'url', 'scores', 'title', 'genres',
                      'boxOffice', and 'rating'.

    Returns: pd.DataFrame with columns for url, criticScore, audienceScore,
    title, boxOffice, rating, and individual binary columns for each specified genre.
    """

    all_genres = [
        "Action", "Adventure", "Animation", "Anime", "Biography", "Comedy", "Crime",
        "Documentary", "Drama", "Entertainment", "Faith & Spirituality", "Fantasy",
        "Game Show", "LGBTQ+", "Health & Wellness", "History", "Holiday", "Horror",
        "House & Garden", "Kids & Family", "Music", "Musical", "Mystery & Thriller",
        "Nature", "News", "Reality", "Romance", "Sci-Fi", "Short", "Soap",
        "Special Interest", "Sports", "Stand-Up", "Talk Show", "Travel", "Variety",
        "War", "Western"
    ]

    df = pd.DataFrame(data_list)

    # 1. Process Scores:
    if 'scores' in df.columns:
        scores_df = pd.json_normalize(df['scores'])

        df = df.join(scores_df)

        df.drop('scores', axis=1, inplace=True)

    if 'criticsScore' not in df.columns:
        df['criticsScore'] = None
    if 'audienceScore' not in df.columns:
        df['audienceScore'] = None

    for score_col in ['criticsScore', 'audienceScore']:
        df[score_col] = df[score_col].replace({'': None, '%': None})
        df[score_col] = df[score_col].astype(str).str.replace('%', '', regex=False)
        df[score_col] = pd.to_numeric(df[score_col], errors='coerce')
        df[score_col] = df[score_col] / 100.0

    # 2. Process boxOffice:
    bo_series = df['boxOffice'].fillna('').astype(str)

    is_million = bo_series.str.contains('M', case=False, na=False)
    is_thousand = bo_series.str.contains('K', case=False, na=False)

    bo_numeric = bo_series.str.replace('$', '', regex=False).str.replace('M', '', regex=False).str.replace('K', '', regex=False)

    bo_numeric = pd.to_numeric(bo_numeric, errors='coerce')

    bo_numeric.loc[is_million] = bo_numeric.loc[is_million] * 1_000_000
    bo_numeric.loc[is_thousand] = bo_numeric.loc[is_thousand] * 1_000

    df['boxOffice'] = bo_numeric

    df['boxOffice'] = df['boxOffice'].fillna(0).astype('int64')

    # 3. Process Rating:
    df['rating'] = df['rating'].fillna('').astype(str)
    df['rating'] = df['rating'].str.split(' ').str[0]
    df['rating'] = df['rating'].replace({'': None})

    # 4. Process Genres (0/1 column for each genre):
    if 'genres' in df.columns:
        df['genres'] = df['genres'].apply(lambda d: d if isinstance(d, list) else [])

        df_dummies = df['genres'].str.join('|').str.get_dummies('|')

        cols_to_add = [col for col in all_genres if col in df_dummies.columns]
        df = df.join(df_dummies[cols_to_add])

        for genre in all_genres:
            if genre not in df.columns:
                df[genre] = 0

        df.drop('genres', axis=1, inplace=True)
    else:
        for genre in all_genres:
            df[genre] = 0

    df = df.dropna()

    return df

In [10]:
from google.colab import files

from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/DS3000/DS3000_Group_Project/')

raw_data_list = json.load(open('data.json', 'r', encoding='utf-8'))

cleaned_df = clean_rotten_tomatoes_data(raw_data_list)

cleaned_df.head(50)

cleaned_df.to_csv("cleaned_data.csv", index=False)

files.download("cleaned_data.csv")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
cleaned_df.head(50)

,url,title,rating,boxOffice,criticsScore,audienceScore,Action,Adventure,Animation,Anime,...,House & Garden,Nature,News,Reality,Short,Soap,Special Interest,Talk Show,Travel,Variety
1,https://www.rottentomatoes.com/m/the_long_walk...,The Long Walk,R,34800000,0.93,0.61,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,https://www.rottentomatoes.com/m/the_roses,The Roses,R,15300000,0.93,0.95,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,https://www.rottentomatoes.com/m/afterburn_2025,Afterburn,R,0,0.79,0.78,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
6,https://www.rottentomatoes.com/m/coyotes_2025,Coyotes,R,0,0.93,0.83,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
16,https://www.rottentomatoes.com/m/the_hand_that...,The Hand That Rocks The Cradle,R,0,0.45,0.30,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
20,https://www.rottentomatoes.com/m/a_house_of_dy...,A HOUSE OF DYNAMITE,R,0,0.95,0.66,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
21,https://www.rottentomatoes.com/m/good_boy_2025,Good Boy,PG-13,6100000,0.95,0.51,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
33,https://www.rottentomatoes.com/m/the_perfect_n...,The Perfect Neighbor,R,0,0.86,0.79,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
34,https://www.rottentomatoes.com/m/weapons,Weapons,R,151500000,0.92,0.70,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
37,https://www.rottentomatoes.com/m/the_substance,The Substance,R,17600000,0.84,0.73,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Part 3:
(2\%) Builds at least two visualizations (graphs/plots) from the data which help to understand or answer the questions of interest. These visualizations will be graded based on how much information they can effectively communicate to readers. Please make sure your visualization are sufficiently distinct from each other.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
df = cleaned_df.copy()

def to_percent(series):
    """
    Converts everything to percent

    Args:
    series (df): data to convert to percent

    Returns: same values from decimals to percent
    """
    s = pd.to_numeric(series, errors="coerce")
    return np.where(s <= 1, s * 100, s)
# Applying the conversion to audience and critic scores
df["audience_pct"] = to_percent(df["audienceScore"])
df["critics_pct"]  = to_percent(df["criticsScore"])
# Defining which columns are not genres
non_genre = {"url", "title", "rating", "boxOffice", "criticsScore", "audienceScore", "audience_pct", "critics_pct"}
genre_cols = [
    c for c in df.columns
    if c not in non_genre
    and df[c].dropna().isin([0,1]).all()
    and df[c].sum() > 0
]
# Converting the data from wide to long format
long = df.melt(
    id_vars=["title","audience_pct","critics_pct","boxOffice"],
    value_vars=genre_cols,
    var_name="genre",
    value_name="is_genre"
)
# Keeping only rows where the movie belongs to that genre
long = long[long["is_genre"] == 1].copy()
# Computing the mean audience score per genre
genre_means = (long.groupby("genre")["audience_pct"]
               .mean()
               .sort_values(ascending=False))
# Bar chart: Average audience score by genre
plt.figure(figsize=(11, 6))
plt.bar(genre_means.index, genre_means.values, edgecolor="black")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Average audience score (%)")
plt.xlabel("Genre")
plt.title("Average audience score by genre")
plt.tight_layout()
plt.show()
# Assigning each movie a "primary" genre (meaning the first genre it appears under)
primary_genre = long.groupby("title")["genre"].first()
# Joining the primary genre back into the main dataframe
df2 = df.join(primary_genre, on="title").rename(columns={"genre":"primary_genre"})
# Computing the marker size for the scatterplot based on the log of box office
size = np.log1p(pd.to_numeric(df2["boxOffice"], errors="coerce").fillna(0))
# Scatter plot: Critics vs audience by title - point size reflects box office
plt.figure(figsize=(8, 6))
plt.scatter(df2["critics_pct"], df2["audience_pct"], s=20 + 10*size, alpha=0.7)
plt.xlabel("Critics score (%)")
plt.ylabel("Audience score (%)")
plt.title("Critics vs audience by title - point size reflects box office")
plt.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()
